In [91]:
import requests 
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time
import csv

In [92]:
all_books = []

In [93]:
for page_num in range(1, 51): 
    page_url = f'https://books.toscrape.com/catalogue/page-{page_num}.html' 
    print(f' Scraping Page {page_num}/50...')
    page_response = requests.get(page_url)
    page_soup = BeautifulSoup(page_response.text, "html.parser")
    books = page_soup.find_all("article", class_="product_pod")
    for book in books:
     title = book.h3.a["title"] 
     price = book.find("p", class_="price_color").text 
     rating_classes = book.find("p", class_="star-rating")["class"] 
     rating_word = [cls for cls in rating_classes if cls != "star-rating"][0] 
     rating_map = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5} 
     rating_number = rating_map.get(rating_word, 0)
     availability_text = book.find("p", class_="instock availability").text.strip()
     is_available = "In stock" in availability_text
     relative_link = book.h3.a["href"] 
     detail_url = urljoin(page_url, relative_link)
     detail_response = requests.get(detail_url)
     detail_soup = BeautifulSoup(detail_response.text, "html.parser")
     breadcrumb_items = detail_soup.find("ul", class_="breadcrumb").find_all("li") 
     category = breadcrumb_items[2].text.strip()
     book_info = { "title": title, "category": category, "rating": rating_number, "price": price, "is_available": is_available, } 
     all_books.append(book_info) 
     print(f"Scraped: {title} | {category} | £{price}| {rating_number} stars | {'In stock' if is_available else 'Out of stock'}")
     time.sleep(0.5)

print(f' COMPLETE! Total books collected: {len(all_books)}')

 Scraping Page 1/50...
Scraped: A Light in the Attic | Poetry | £Â£51.77| 3 stars | In stock
Scraped: Tipping the Velvet | Historical Fiction | £Â£53.74| 1 stars | In stock
Scraped: Soumission | Fiction | £Â£50.10| 1 stars | In stock
Scraped: Sharp Objects | Mystery | £Â£47.82| 4 stars | In stock
Scraped: Sapiens: A Brief History of Humankind | History | £Â£54.23| 5 stars | In stock
Scraped: The Requiem Red | Young Adult | £Â£22.65| 1 stars | In stock
Scraped: The Dirty Little Secrets of Getting Your Dream Job | Business | £Â£33.34| 4 stars | In stock
Scraped: The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull | Default | £Â£17.93| 3 stars | In stock
Scraped: The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics | Default | £Â£22.60| 4 stars | In stock
Scraped: The Black Maria | Poetry | £Â£52.15| 1 stars | In stock
Scraped: Starving Hearts (Triangular Trade Trilogy, #1) | Default | £Â£13.99| 2 stars | In sto

In [94]:
with open("books_catalog.csv", "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(
        file,
        fieldnames=["title", "category", "rating", "price", "is_available"]
    )
    writer.writeheader()
    writer.writerows(all_books)